In [ ]:
import cv2
import numpy as np

def classify_coin(radius, image, center):
    """
    Classify a coin based on its detected radius and average color inside the coin.
    The thresholds below are heuristic and may need tuning for your images.
    """
    # Create a mask for the coin region
    mask = np.zeros(image.shape[:2], dtype=np.uint8)
    cv2.circle(mask, center, int(radius), 255, -1)
    avg_color = cv2.mean(image, mask=mask)  # Returns (B, G, R, alpha)

    # Classification based on radius (in pixels)
    # Adjust these thresholds based on calibration with a reference image.
    if radius < 23:
        return "Dime"
    elif radius < 28:
        # Use color information to differentiate between penny (copper-colored) and nickel.
        # Pennies tend to have a higher red channel compared to blue.
        if avg_color[2] > avg_color[0] + 20:
            return "Penny"
        else:
            return "Nickel"
    elif radius < 32:
        return "Nickel"
    else:
        return "Quarter"

def process_image(input_filename, output_filename):
    # Load the image of coins.
    img = cv2.imread(input_filename)
    if img is None:
        print(f"Error: {input_filename} not found. Ensure the image is in the correct location.")
        return

    # Convert to grayscale and apply a Gaussian blur to reduce noise.
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    # Use the Hough Circle Transform to detect circles (coins).
    circles = cv2.HoughCircles(
        blur,
        cv2.HOUGH_GRADIENT,
        dp=1.2,
        minDist=50,
        param1=50,
        param2=30,
        minRadius=20,
        maxRadius=100
    )

    # If coins are detected, classify and annotate them.
    if circles is not None:
        circles = np.uint16(np.around(circles))
        for i in circles[0, :]:
            center = (i[0], i[1])
            radius = i[2]
            coin_value = classify_coin(radius, img, center)
            
            # Draw the detected coin circle and its center.
            cv2.circle(img, center, radius, (0, 255, 0), 2)
            cv2.circle(img, center, 2, (0, 0, 255), 3)
            
            # Annotate the coin with its determined value.
            cv2.putText(img, coin_value, (center[0] - 40, center[1]),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
            print(f"Detected coin at {center} with radius {radius} pixels: {coin_value}")
    else:
        print("No coins detected.")

    # Save the annotated image to a file.
    cv2.imwrite(output_filename, img)
    print(f"Annotated image saved as '{output_filename}'.")

process_image("synthetic_coins.jpg", "annotated_coins.jpg")


Detected coin at (301, 301) with radius 36 pixels: Quarter
Detected coin at (100, 100) with radius 22 pixels: Dime
Detected coin at (100, 298) with radius 29 pixels: Nickel
Detected coin at (301, 100) with radius 23 pixels: Penny
Annotated image saved as 'annotated_coins.jpg'.


In [8]:
import cv2
import numpy as np

def classify_coin(radius, image, center):
    """
    Classify a coin based on its radius and average color.
    Adjust the thresholds to match your specific image.
    """
    # Create a mask for the coin region to calculate the average color.
    mask = np.zeros(image.shape[:2], dtype=np.uint8)
    cv2.circle(mask, center, int(radius), 255, -1)
    avg_b, avg_g, avg_r, _ = cv2.mean(image, mask=mask)
    
    # Debug print (optional): uncomment to see measured values
    # print(f"[DEBUG] center={center}, radius={radius:.2f}, avg_color=(B={avg_b:.1f}, G={avg_g:.1f}, R={avg_r:.1f})")
    
    # Example classification based on measured radius (in pixels)
    if radius < 23:
        return "Dime"
    elif radius < 26:
        # Use color info to differentiate: pennies have a higher red channel.
        if avg_r > avg_b + 20:
            return "Penny"
        else:
            return "Dime"
    elif radius < 29:
        return "Nickel"
    else:
        return "Quarter"

def process_image(input_filename, output_filename):
    # Load the image of coins
    img = cv2.imread(input_filename)
    if img is None:
        print(f"Error: Could not load the image '{input_filename}'.")
        return

    # Convert to grayscale and apply a Gaussian blur to reduce noise.
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (7, 7), 0)

    # Use HoughCircles to detect circles (coins)
    circles = cv2.HoughCircles(
        blur,
        cv2.HOUGH_GRADIENT,
        dp=1.0,      # Adjust as needed (1.2 or 1.5 might work better in some cases)
        minDist=40,  # Minimum distance between coin centers; adjust based on your image
        param1=50,   # Higher threshold for Canny edge detector
        param2=30,   # Accumulator threshold for circle detection (lower -> more circles)
        minRadius=15,  # Minimum coin radius in pixels
        maxRadius=100  # Maximum coin radius in pixels
    )

    if circles is not None:
        circles = np.uint16(np.around(circles))
        for circle in circles[0, :]:
            center = (circle[0], circle[1])
            radius = circle[2]
            
            # Classify each coin based on radius and color.
            coin_type = classify_coin(radius, img, center)
            
            # Draw the coin's outline and center.
            cv2.circle(img, center, radius, (0, 255, 0), 2)
            cv2.circle(img, center, 2, (0, 0, 255), 3)
            
            # Annotate the coin with its classification.
            cv2.putText(img, coin_type, (center[0] - 40, center[1]),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)
            print(f"Detected coin at {center} -> {coin_type}, radius={radius}")
    else:
        print("No coins detected. Try adjusting the detection parameters.")

    # Save the annotated image to a file
    cv2.imwrite(output_filename, img)
    print(f"Annotated image saved as '{output_filename}'.")

if __name__ == "__main__":
    input_image = "coin_2.jpeg"         # Replace with your input image filename
    output_image = "annotated_coins.jpg"  # Output filename for the annotated image
    process_image(input_image, output_image)


Detected coin at (140, 164) -> Quarter, radius=58
Detected coin at (304, 382) -> Quarter, radius=43
Detected coin at (138, 304) -> Quarter, radius=50
Detected coin at (302, 168) -> Quarter, radius=45
Detected coin at (308, 266) -> Quarter, radius=41
Detected coin at (296, 484) -> Quarter, radius=40
Detected coin at (220, 204) -> Quarter, radius=97
Detected coin at (188, 238) -> Quarter, radius=99
Detected coin at (256, 224) -> Quarter, radius=80
Detected coin at (122, 416) -> Quarter, radius=37
Detected coin at (146, 362) -> Quarter, radius=47
Detected coin at (126, 254) -> Quarter, radius=49
Detected coin at (166, 198) -> Dime, radius=16
Detected coin at (116, 130) -> Dime, radius=18
Detected coin at (170, 136) -> Dime, radius=17
Detected coin at (170, 274) -> Dime, radius=25
Detected coin at (116, 204) -> Dime, radius=25
Annotated image saved as 'annotated_coins.jpg'.
